In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
# 1. Install and Import Necessary Libraries
# -------------------------------------------
# Using the same libraries as the first agent for consistency.
!pip install -q transformers accelerate torchaudio librosa
!pip install torchcodec==0.2
!pip install fsspec==2023.9.2
!pip install datasets==2.10.0
!pip install evaluate==0.4.0
!pip install jiwer
!pip install tf-keras

!pip install --user --no-cache-dir pyarrow==15.0.2

import os
import re
import json
from dataclasses import dataclass
from typing import Dict, List, Optional, Union

import torch
import torch.nn as nn
import torchaudio
import numpy as np
from torch.nn.modules.utils import _pair # Needed for custom Conv2d

from datasets import load_dataset, Audio, DatasetDict
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ConformerForCTC,
    TrainingArguments,
    Trainer
)
import evaluate
#%pip install -U datasets



In [ ]:
!pip install jiwer

In [2]:
# 2. Configuration Variables
# --------------------------
# Dataset related (UC-DM-01)
DATASET_NAME = "aconeil/nchlt"
DEFAULT_CONFIG_NAME = "default"
TEXT_COLUMN = "transcription"
AUDIO_COLUMN = "audio"

# Model related (UC-AD-01)
#MODEL_NAME = "facebook/wav2vec2-conformer-large-960h"
MODEL_NAME = "MubarakB/wav2vec2-large-xls-r-300m-zulu-vocab"
MODEL_OUTPUT_DIR = "/content/drive/MyDrive/wav2vecconformer-finetune-checkpoints"
TOKENIZER_OUTPUT_DIR = "/content/drive/MyDrive/wav2vecconformer-finetune-tokenizer" # for vocabulary

# Training parameters (UC-AD-02)
TARGET_SAMPLING_RATE = 16_000
MAX_DURATION_IN_SECONDS = 20.0 # Filter out very long audio samples to avoid OOM errors
MIN_DURATION_IN_SECONDS = 1.0  # Filter out very short audio samples
NUM_EPOCHS = 10 # Start with a few epochs for prototyping
PER_DEVICE_TRAIN_BATCH_SIZE = 2
PER_DEVICE_EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 2 # Effective batch size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
LEARNING_RATE = 3e-4
WARMUP_STEPS = 5
SAVE_STEPS = 10
EVAL_STEPS = 10

In [3]:
# Forcefully clear Hugging Face datasets and general cache
!rm -rf ~/.cache/huggingface/datasets/*
!rm -rf ~/.cache/huggingface/modules/*
!rm -rf ~/.cache/huggingface/hub/*
print("Hugging Face caches cleared.")

Hugging Face caches cleared.


In [4]:
# 3. Load and Prepare Dataset (UC-DM-01)
# -------------------------------------------------
print(f"Loading dataset: {DATASET_NAME}, config: {DEFAULT_CONFIG_NAME}")
try:
    raw_datasets = DatasetDict()
    # Load train and test splits
    train_ds = load_dataset(DATASET_NAME, name=DEFAULT_CONFIG_NAME, split="train")
    test_ds = load_dataset(DATASET_NAME, name=DEFAULT_CONFIG_NAME, split="test")

    # This dataset doesn't have a pre-defined validation split.
    # We will create one from the training data.
    print("Train and test splits loaded. Creating validation split from training data...")
    # Split the training data (90% train, 10% validation)
    train_val_split = train_ds.train_test_split(test_size=0.1, shuffle=True, seed=42) # Use a seed for reproducibility

    #raw_datasets["train"] = train_val_split["train"]
    #raw_datasets["validation"] = train_val_split["test"] # The 'test' part of this split becomes our validation set
    #raw_datasets["test"] = test_ds # Keep the original test set

    #print("Dataset loaded and split successfully.")
    #print(raw_datasets)
    #print(f"\nFeatures of the training set: {raw_datasets['train'].features}")
    #if len(raw_datasets['train']) > 0:
        #print(f"First example from the training set: {raw_datasets['train'][0]}")

    # Define the size of your subset
    TRAIN_SUBSET_SIZE = 100  # Number of training examples
    EVAL_SUBSET_SIZE = 100   # Number of validation/test examples

    raw_datasets["train"] = train_val_split["train"].select(range(TRAIN_SUBSET_SIZE))
    raw_datasets["validation"] = train_val_split["test"].select(range(EVAL_SUBSET_SIZE))
    raw_datasets["test"] = train_val_split["test"].select(range(EVAL_SUBSET_SIZE))

    print("Subset selected successfully:")
    print(raw_datasets)

except Exception as e:
    print(f"Error loading dataset: {e}")
    from datasets import get_dataset_config_names
    try:
        configs = get_dataset_config_names(DATASET_NAME)
        print(f"Available configs for {DATASET_NAME}: {configs}")
    except Exception as e_configs:
        print(f"Could not retrieve configs for {DATASET_NAME}: {e_configs}")
    raise e

Loading dataset: aconeil/nchlt, config: default


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating test split:   0%|          | 0/2802 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/41871 [00:00<?, ? examples/s]

Dataset parquet downloaded and prepared to /root/.cache/huggingface/datasets/aconeil___parquet/aconeil--nchlt-be2e393fe3a31823/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec. Subsequent calls will reuse this data.


/usr/local/lib/python3.12/dist-packages/datasets/table.py:1427: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


Train and test splits loaded. Creating validation split from training data...
Subset selected successfully:
DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 100
    })
    test: Dataset({
        features: ['audio', 'transcription', 'gender', 'age', 'speaker_id', 'duration', 'md5sum', 'pdp_score'],
        num_rows: 100
    })
})


In [5]:
# --- Text Preprocessing --- (UC-DM-02)
# Remove special characters, keep isiZulu characters and spaces
chars_to_remove_regex = r'[!"#$%&\'()*+,-./:;<=>?@\[\\\]^_`{|}~0-9]'

def remove_special_characters(batch):
    # Ensure the text column exists and is not None
    if batch[TEXT_COLUMN] is not None:
        batch[TEXT_COLUMN] = re.sub(chars_to_remove_regex, '', batch[TEXT_COLUMN]).lower()
    else:
        # Handle cases where the text might be missing, e.g., by providing an empty string
        batch[TEXT_COLUMN] = ""
    return batch

print("Normalizing text data...")
raw_datasets = raw_datasets.map(remove_special_characters, num_proc=1)

Normalizing text data...


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [6]:
# --- Create Vocabulary ---
# For Mapping each unique character present in the text to a numerical ID

#def extract_all_chars(batch):
  #all_text = " ".join(batch[TEXT_COLUMN])
  #vocab = list(set(all_text))
  #return {"vocab": [vocab], "all_text": [all_text]}

print("Extracting vocabulary...")
# Concatenate all splits to build a comprehensive vocabulary
combined_text_data = "".join([text for split in raw_datasets for text in raw_datasets[split][TEXT_COLUMN] if text is not None])
vocab_list = list(set(combined_text_data))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}
vocab_dict["|"] = vocab_dict[" "] # CTC blank token representation
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict) # Unknown token
vocab_dict["[PAD]"] = len(vocab_dict) # Padding token for tokenizer (CTC doesn't use it for loss)
print(f"Vocabulary size: {len(vocab_dict)}")


Extracting vocabulary...
Vocabulary size: 29


In [7]:
# Save vocabulary
os.makedirs(TOKENIZER_OUTPUT_DIR, exist_ok=True)
with open(os.path.join(TOKENIZER_OUTPUT_DIR, 'vocab.json'), 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

In [8]:
# --- Load Tokenizer and Feature Extractor ---
# Set up the necessary tools, a tokenizer and a feature extractor, to prepare both the text transcripts and the audio data for the Wav2Vec2 model.
# (UC-AD-01)
tokenizer = Wav2Vec2CTCTokenizer(
    os.path.join(TOKENIZER_OUTPUT_DIR, 'vocab.json'),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=TARGET_SAMPLING_RATE,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=False # CTC doesn't use attention mask for features
)

In [9]:
# --- Audio Preprocessing ---
# (UC-DM-02)
# Ensure audio is 16kHz mono
print("Casting audio column to 16kHz Audio feature...")
raw_datasets = raw_datasets.cast_column(
    AUDIO_COLUMN, Audio(sampling_rate=TARGET_SAMPLING_RATE)
)

Casting audio column to 16kHz Audio feature...


In [10]:
# Filter audio by duration
def filter_duration(example):
    if example[AUDIO_COLUMN] is None or example[AUDIO_COLUMN]["array"] is None:
        return False
    duration = len(example[AUDIO_COLUMN]["array"]) / example[AUDIO_COLUMN]["sampling_rate"]
    return MIN_DURATION_IN_SECONDS <= duration <= MAX_DURATION_IN_SECONDS

print(f"Filtering audio by duration ({MIN_DURATION_IN_SECONDS}s - {MAX_DURATION_IN_SECONDS}s)...")
initial_sizes = {split: len(raw_datasets[split]) for split in raw_datasets}
raw_datasets = raw_datasets.filter(filter_duration, num_proc=os.cpu_count())
filtered_sizes = {split: len(raw_datasets[split]) for split in raw_datasets}

for split in initial_sizes:
    print(f"Split '{split}': {initial_sizes[split]} -> {filtered_sizes[split]} samples after duration filtering.")


Filtering audio by duration (1.0s - 20.0s)...


Filter (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

Filter (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

Split 'train': 100 -> 100 samples after duration filtering.
Split 'validation': 100 -> 100 samples after duration filtering.
Split 'test': 100 -> 100 samples after duration filtering.


In [11]:
from transformers import AutoProcessor, AutoModelForCTC

processor = AutoProcessor.from_pretrained("MubarakB/wav2vec2-large-xls-r-300m-zulu-vocab")
model = AutoModelForCTC.from_pretrained("MubarakB/wav2vec2-large-xls-r-300m-zulu-vocab")

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

vocab.json:   0%|          | 0.00/512 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

In [12]:
#Transform the raw audio and text data into a format that can be understood and processed by the Wav2Vec2 model.

def prepare_dataset(batch):
    audio = batch[AUDIO_COLUMN]
    # batched output is "un-batched"
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    with processor.as_target_processor():
        batch["labels"] = processor(batch[TEXT_COLUMN]).input_ids
    return batch

print("Preparing dataset with processor...")
processed_datasets = raw_datasets.map(
    prepare_dataset,
    remove_columns=raw_datasets.column_names["train"], # remove original columns
    num_proc= 1 # use num_proc for faster processing
)
print("Dataset preparation complete.")

Preparing dataset with processor...


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset preparation complete.


In [13]:
RAW_TEXT_COLUMN = "text"

# The dataset split you want to check (e.g., "train", "test")
SPLIT = "test"
# --------------------
import random
def verify_random_sample(raw_datasets, processed_datasets, processor):
    """
    Selects a random sample and compares its raw text to its processed label.
    """
    print(f"--- Verifying a random sample from the '{SPLIT}' split ---")

    # 1. Select a random index
    try:
        num_samples = len(processed_datasets[SPLIT])
        random_index = random.randint(0, num_samples - 1)
        print(f"Checking sample at index: {random_index}")
    except Exception as e:
        print(f"Could not get random sample. Error: {e}")
        return

    # 2. Get the corresponding samples from both datasets
    processed_sample = processed_datasets[SPLIT][random_index]
    raw_sample = raw_datasets[SPLIT][random_index]

    # 3. Get the original text from the raw dataset
    try:
        original_text = raw_sample[TEXT_COLUMN]
    except KeyError:
        print(f"ERROR: Column '{TEXT_COLUMN}' not found in the raw dataset.")
        print(f"Available columns are: {list(raw_sample.keys())}")
        return

    # 4. Decode the processed labels back to text
    labels = torch.tensor(processed_sample["labels"]).unsqueeze(0)
    decoded_processed_text = processor.batch_decode(labels, group_tokens=False)[0]

    # 5. Print for comparison
    print("\nOriginal Text from Raw File:")
    print(f"==> \"{original_text}\"")

    print("\nProcessed Label (Decoded):")
    print(f"==> \"{decoded_processed_text}\"")


# --- Run the verification ---
# Make sure your datasets and processor are loaded before calling this
verify_random_sample(raw_datasets, processed_datasets, processor)

--- Verifying a random sample from the 'test' split ---
Checking sample at index: 19

Original Text from Raw File:
==> "ukuqinisekisa ukulandelwa kwemithetho"

Processed Label (Decoded):
==> "ukuqinisekisa ukulandelwa kwemithetho"


In [ ]:
# 9. Test the model with a single example
# ---------------------------------------
print("Testing the model with a single example from the test set...")

# Select a single example from the test dataset
sample = processed_datasets["test"][0]

# Get the input values and labels
input_values = torch.tensor(sample["input_values"]).unsqueeze(0)
labels = torch.tensor(sample["labels"]).unsqueeze(0)

# Move the model to evaluation mode and to the appropriate device
model.eval()
if torch.cuda.is_available():
    model.to("cuda")
    input_values = input_values.to("cuda")
    labels = labels.to("cuda")


# Perform inference
with torch.no_grad():
    logits = model(input_values).logits

# Get the predicted token IDs
predicted_ids = torch.argmax(logits, dim=-1)

# Decode the predicted token IDs to text
predicted_sentence = processor.batch_decode(predicted_ids)[0]

# Decode the ground truth labels to text
ground_truth_sentence = processor.batch_decode(labels, group_tokens=False)[0]

print(f"Ground Truth: {ground_truth_sentence}")
print(f"Predicted: {predicted_sentence}")

Testing the model with a single example from the test set...
Ground Truth: okuphuzi ngokujiyile s okunombala
Predicted: 


In [14]:
# 5. Define Training Components (UC-AD-02)
# ----------------------------------------
# Data Collator
# This is responsible for taking a list of individual data samples and combining them into a batch that can be fed into the model during training.
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lenghts and need different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [15]:
# Evaluation Metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    # replace -100 with pad_token_id
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)

    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)   # we do not want to group tokens when computing the metrics

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

ImportError: cannot import name 'compute_measures' from 'jiwer' (/usr/local/lib/python3.12/dist-packages/jiwer/__init__.py)

In [ ]:
WARMUP_STEPS = 50
SAVE_STEPS = 100
EVAL_STEPS = 100

# Training Arguments
training_args = TrainingArguments(
  output_dir=MODEL_OUTPUT_DIR,
  group_by_length=True, # Speeds up training by batching similar length audio
  per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
  per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
  gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
  eval_strategy="steps",
  num_train_epochs=100,
  fp16=True,
  save_steps=SAVE_STEPS,
  eval_steps=EVAL_STEPS,
  logging_steps=EVAL_STEPS // 2,
  learning_rate=LEARNING_RATE,
  warmup_steps=WARMUP_STEPS,
  save_total_limit=2,
  load_best_model_at_end=True,
  metric_for_best_model="wer",
  greater_is_better=False,
)


In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["validation"],
    tokenizer=processor.feature_extractor, #pass the feature_extractor for Wav2Vec2
)


NameError: name 'compute_metrics' is not defined

In [ ]:
# 6. Start Fine-tuning (UC-AD-02)
# --------------------------------
print("Starting fine-tuning...")
try:
    trainer.train()
    print("Fine-tuning finished.")
except Exception as e:
    print(f"Error during training: {e}")
    print("This might be due to OOM errors. Try reducing batch size or gradient accumulation steps.")
    raise e

Starting fine-tuning...


/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


Step,Training Loss,Validation Loss,Wer,Cer
100,3.557200,3.067366,1.000000,1.000000
200,3.063700,3.048058,1.000000,1.000000
300,3.033200,3.064492,1.000000,1.000000
400,2.937600,2.959605,1.000000,1.000000
500,2.820000,2.823623,1.000000,1.000000
600,2.623300,2.674044,1.000000,0.864095
700,2.348500,2.428275,1.000000,0.735040
800,1.966600,2.381174,1.006689,0.686734
900,1.527500,2.430385,1.046823,0.600937
1000,1.128700,2.469306,1.063545,0.567412


/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/wav2vec2/processing_wav2vec2.py:188: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call

Fine-tuning finished.


In [ ]:
# 8. Test the Saved Model with a Single Example
# ---------------------------------------------
print("Testing the SAVED model with a single example from the test set...")

# Load the saved model and processor
try:
    # Load the model from the saved checkpoint
    loaded_model = Wav2Vec2ConformerForCTC.from_pretrained(MODEL_OUTPUT_DIR)
    # Load the processor (tokenizer and feature extractor) from the saved directory
    # Fix: Load the processor using the original pretrained model identifier
    loaded_processor = AutoProcessor.from_pretrained(MODEL_NAME)
    print("Saved model and processor loaded successfully.")
except Exception as e:
    print(f"Error loading saved model or processor: {e}")
    print(f"Please ensure the directory {MODEL_OUTPUT_DIR} exists and contains the necessary files.")
    raise e


# Select a single example from the test dataset
# Ensure processed_datasets is available
if 'processed_datasets' not in locals():
    print("Error: 'processed_datasets' not found. Please run the data processing steps first.")
else:
    try:
        sample = processed_datasets["train"][1]

        # Get the input values and labels
        input_values = torch.tensor(sample["input_values"]).unsqueeze(0)
        labels = torch.tensor(sample["labels"]).unsqueeze(0)

        # Move the loaded model to evaluation mode and to the appropriate device
        loaded_model.eval()
        if torch.cuda.is_available():
            loaded_model.to("cuda")
            input_values = input_values.to("cuda")
            labels = labels.to("cuda")


        # Perform inference with the loaded model
        with torch.no_grad():
            logits = loaded_model(input_values).logits

        # Get the predicted token IDs
        predicted_ids = torch.argmax(logits, dim=-1)

        # Decode the predicted token IDs to text using the loaded processor
        predicted_sentence = loaded_processor.batch_decode(predicted_ids)[0]

        # Decode the ground truth labels to text using the loaded processor
        ground_truth_sentence = loaded_processor.batch_decode(labels, group_tokens=False)[0]

        print(f"\nGround Truth: {ground_truth_sentence}")
        print(f"Predicted (from saved model): {predicted_sentence}")

    except Exception as e:
        print(f"Error during inference on a single sample: {e}")

Testing the SAVED model with a single example from the test set...
Saved model and processor loaded successfully.

Ground Truth: esiphelele sokuzalwa isitifikedi
Predicted (from saved model): e s iu phe l e l e s o ku z a lwa i s i thi bpi k e d i


In [ ]:
# 9. Test the Saved Model with a Specific Audio File
# ---------------------------------------------------
print("Testing the SAVED model with a specific audio file...")

# Define the path to the audio file
audio_file_path = "/content/test2.wav" # Assuming the file is in the content directory

# Load the audio file
try:
    # Load the audio file and resample if necessary
    speech_array, sampling_rate = torchaudio.load(audio_file_path)
    # Ensure the sampling rate matches the model's expected sampling rate
    if sampling_rate != loaded_processor.feature_extractor.sampling_rate:
        resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=loaded_processor.feature_extractor.sampling_rate)
        speech_array = resampler(speech_array)
    # Squeeze the tensor to remove the channel dimension if it's mono
    speech_array = speech_array.squeeze(0)
    print(f"Audio file '{audio_file_path}' loaded successfully.")
except Exception as e:
    print(f"Error loading audio file '{audio_file_path}': {e}")
    print("Please ensure the file exists and is a valid audio file.")
    raise e

# Preprocess the audio file
try:
    # Process the audio data to get input values for the model
    input_values = loaded_processor(speech_array, sampling_rate=loaded_processor.feature_extractor.sampling_rate, return_tensors="pt").input_values

    # Move input values to the appropriate device
    if torch.cuda.is_available():
        input_values = input_values.to("cuda")

    print("Audio preprocessing complete.")
except Exception as e:
    print(f"Error during audio preprocessing: {e}")
    raise e

# Perform inference with the loaded model
try:
    loaded_model.eval() # Set the model to evaluation mode
    with torch.no_grad():
        logits = loaded_model(input_values).logits

    # Get the predicted token IDs
    predicted_ids = torch.argmax(logits, dim=-1)

    # Decode the predicted token IDs to text using the loaded processor
    predicted_sentence = loaded_processor.batch_decode(predicted_ids)[0]

    print(f"\nPredicted Transcription: {predicted_sentence}")
except Exception as e:
    print(f"Error during inference: {e}")
    raise e

Testing the SAVED model with a specific audio file...
Audio file '/content/test2.wav' loaded successfully.
Audio preprocessing complete.

Predicted Transcription: u tj e s u wa hl a l a e z we n l a kwa i z d a ye


In [ ]:
# 7. Evaluate the Model on the Test Set (UC-AD-03)
# ------------------------------------------------
print("Evaluating the model on the test set...")

try:
    # Load the best model saved during training
    # The trainer saves the best model based on the metric specified in training_args
    best_model_checkpoint = trainer.state.best_model_checkpoint
    if best_model_checkpoint:
        print(f"Loading best model from: {best_model_checkpoint}")
        # Use the loaded_model from the previous test cell if available, otherwise load
        if 'loaded_model' not in locals() or loaded_model is None:
             loaded_model = Wav2Vec2ConformerForCTC.from_pretrained(best_model_checkpoint)
             # Ensure the loaded model is on the correct device
             if torch.cuda.is_available():
                 loaded_model.to("cuda")
        else:
             print("Using already loaded best model.")

        # Set the trainer's model to the best loaded model for evaluation
        trainer.model = loaded_model

        # Evaluate the model on the test dataset
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["test"])

        print("\nTest set evaluation results:")
        print(eval_results)

    else:
        print("No best model checkpoint found. Please ensure training completed successfully and save_total_limit > 0.")

except Exception as e:
    print(f"Error during evaluation on the test set: {e}")
    raise e

Evaluating the model on the test set...
Error during evaluation on the test set: name 'trainer' is not defined


NameError: name 'trainer' is not defined